# 14 · Стереозрение и 3D-реконструкция / Stereo vision and 3D reconstruction
**RU.** Реальная пара Middlebury 2014 Motorcycle: два ректифицированных RGB-изображения 500×741 и эталонная disparity с NaN/inf там, где GT отсутствует. Наблюдение — соответствующие пиксели двух камер; признаки — яркость и локальная текстура; цель — disparity, затем метрические координаты 3D. [Источник Middlebury](https://vision.middlebury.edu/stereo/data/scenes2014/), [официальная копия scikit-image](https://scikit-image.org/docs/stable/api/skimage.data.html#skimage.data.stereo_motorcycle). Middlebury разрешает использование и публикацию с цитированием Scharstein et al., GCPR 2014.
**EN.** Real Middlebury 2014 Motorcycle pair: two rectified 500×741 RGB images and reference disparity with nonfinite values for unavailable GT. Observations are corresponding camera pixels, features are intensity and local texture, targets are disparity and metric 3D coordinates. The scikit-image copy downsamples the original by four. Middlebury permits use/publication with citation of Scharstein et al., GCPR 2014.

**RU.** CPU, около 2.5 МБ загрузок, закреплённые SHA256. Единственная сцена, параметры SGBM заданы до оценки: это воспроизводимая демонстрация, а не test обобщения на новые сцены. Пиксели пространственно зависимы, случайное разбиение не создаёт независимый test. Для сравнения моделей нужны дополнительные целиком отложенные сцены. Истинная disparity нужна только оценщику.
**EN.** CPU, about 2.5 MB of downloads with pinned SHA256 hashes. One scene with SGBM settings fixed before evaluation: a reproducible demonstration, not evidence of generalization. Spatially dependent pixels do not become an independent test through random splitting. Model comparison requires additional held-out scenes. Ground-truth disparity is used only by evaluation.

**Калибровка / Calibration.** f=994.978 px, cx=311.193 px, cy=254.877 px, doffs=cx_right−cx_left=31.086 px, B=193.001 mm=0.193001 m. Эти значения относятся к уменьшенным изображениям. / These values apply to the downsampled images. Z=fB/(d+doffs), X=(u−cx)Z/f, Y=(v−cy)Z/f. Camera coordinates: X right, Y down, Z forward.

## ДЗ / Homework
RU: Четыре функции по 25 баллов. Заполните имя/группу, выполните весь ноутбук и сдайте .ipynb с выводами. Срок — в матрице по группе, штрафы применяет внешний проверяющий. EN: Four functions worth 25 points each. Fill identity/group, run all cells, submit the notebook with outputs. The group matrix defines deadlines; the external grader applies penalties.

In [ ]:
student_first=""  # RU: укажите имя; EN: enter first name.
student_last=""  # RU: укажите фамилию; EN: enter last name.
student_group=""  # RU: укажите группу; EN: enter group.
assignment_id="ST3D-14"  # RU: сохраняйте ID задания; EN: preserve the assignment ID.

In [ ]:
from pathlib import Path  # RU: управляем локальным кэшем; EN: manage the local cache.
import urllib.request,hashlib,json  # RU: загружаем и проверяем данные; EN: download and verify data.
import numpy as np  # RU: вычисляем геометрию; EN: compute geometry.
import cv2  # RU: сопоставляем стереопару; EN: match the stereo pair.
import matplotlib.pyplot as plt  # RU: визуализируем измерения; EN: visualize measurements.
data=next((p for p in [Path('build/data'),Path('../build/data'),Path('data')] if (p/'motorcycle_left.png').exists()),Path('data'))  # RU: переиспользуем кэш; EN: reuse the cache.
data.mkdir(parents=True,exist_ok=True)  # RU: создаём каталог при необходимости; EN: create a directory if needed.
hashes={'motorcycle_left.png':'db18e9c4157617403c3537a6ba355dfeafe9a7eabb6b9b94cb33f6525dd49179','motorcycle_right.png':'5fc913ae870e42a4b662314bc904d1786bcad8e2f0b9b67dba5a229406357797','motorcycle_disp.npz':'2e49c8cebff3fa20359a0cc6880c82e1c03bbb106da81a177218281bc2f113d7'}  # RU: закрепляем официальный реестр scikit-image; EN: pin the official scikit-image registry.
for name,digest in hashes.items():  # RU: проверяем каждый ресурс; EN: verify each resource.
    path=data/name  # RU: задаём путь файла; EN: set the file path.
    if not path.exists(): urllib.request.urlretrieve('https://raw.githubusercontent.com/scikit-image/scikit-image/v0.25.2/skimage/data/'+name,path)  # RU: скачиваем версионированную копию; EN: download a versioned copy.
    assert hashlib.sha256(path.read_bytes()).hexdigest()==digest  # RU: отклоняем изменённые данные; EN: reject changed data.
left=cv2.cvtColor(cv2.imread(str(data/'motorcycle_left.png')),cv2.COLOR_BGR2RGB)  # RU: читаем левое RGB-изображение; EN: read the left RGB image.
right=cv2.cvtColor(cv2.imread(str(data/'motorcycle_right.png')),cv2.COLOR_BGR2RGB)  # RU: читаем правое RGB-изображение; EN: read the right RGB image.
gt=np.load(data/'motorcycle_disp.npz')['arr_0']  # RU: загружаем эталонные смещения; EN: load reference disparities.
calib={'f':994.978,'cx':311.193,'cy':254.877,'doffs':31.086,'baseline_m':.193001}  # RU: используем калибровку четверти разрешения; EN: use quarter-resolution calibration.
matcher=cv2.StereoSGBM_create(minDisparity=0,numDisparities=128,blockSize=5,P1=8*25,P2=32*25,disp12MaxDiff=1,uniquenessRatio=10,speckleWindowSize=50,speckleRange=2,mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY)  # RU: фиксируем настройки до оценки; EN: fix settings before evaluation.
raw=matcher.compute(cv2.cvtColor(left,cv2.COLOR_RGB2GRAY),cv2.cvtColor(right,cv2.COLOR_RGB2GRAY))  # RU: вычисляем disparity в фиксированной точке; EN: compute fixed-point disparity.
assert left.shape==right.shape==(500,741,3) and gt.shape==(500,741)  # RU: проверяем согласование изображений и GT; EN: verify image and GT alignment.


## 1 · Масштаб калибровки / Calibration scaling
RU: `scale_calibration(calibration,sx,sy)` возвращает новый dict fx,fy,cx,cy,doffs,baseline_m. Исходные fx/fy,cx/cy умножаются на sx/sy, doffs на sx, baseline_m сохраняется. sx,sy конечные положительные, иначе ValueError. Модель изменения координат u_new=sx*u_old, v_new=sy*v_old без дополнительного crop. EN: Scale intrinsics under the stated coordinate map, preserving physical baseline. Return a new dictionary and reject invalid scale factors.

In [ ]:
def scale_calibration(calibration,sx,sy):  # RU: согласуем калибровку с размером изображения; EN: align calibration with image scaling.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте функцию; EN: implement the function.

## 2 · Общая доля ошибок / Overall bad-pixel rate
RU: `bad_pixel_report(pred,truth,tolerance)` возвращает gt_count,valid_count,bad_count,bad_fraction. GT: finite truth>0, prediction: finite pred>0. Отсутствующие predictions и |pred−truth|>tolerance считаются плохими. Порог включителен для хороших пикселей; no GT: fraction=NaN. Несовпадающие формы, отрицательный/неконечный tolerance — ValueError. EN: Count missing predictions and errors strictly larger than tolerance as bad over all positive finite GT. Validate shapes and tolerance; return NaN fraction without GT.

In [ ]:
def bad_pixel_report(pred,truth,tolerance):  # RU: учитываем точность и пропущенные ответы; EN: count errors and missing answers.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте функцию; EN: implement the function.

## 3 · Ректифицированная триангуляция / Rectified triangulation
RU: `triangulate_rectified(uv_left,uv_right,calibration)` получает соответствия (N,2), calibration содержит fx,fy,cx,cy,doffs,baseline_m. d=u_left−u_right, Z=fx*B/(d+doffs), X=(u_left−cx)Z/fx, Y=(v_left−cy)Z/fy. Неконечные координаты или d+doffs<=0 дают строку NaN. Формы должны совпадать (N,2), fx/fy/B>0, иначе ValueError. Предполагаются ректификация и уже проверенное соответствие строк. EN: Triangulate corresponding rectified pixels with distinct principal-point offset. Return XYZ in metres; mark invalid correspondences with NaN and validate shapes/intrinsics.

In [ ]:
def triangulate_rectified(uv_left,uv_right,calibration):  # RU: восстанавливаем XYZ из пары лучей; EN: recover XYZ from paired rays.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте функцию; EN: implement the function.

## 4 · Ошибка репроекции / Reprojection error
RU: `reprojection_errors(points,pixels,calibration)` возвращает вектор евклидовых ошибок в пикселях для левой камеры: u=fx*X/Z+cx, v=fy*Y/Z+cy. Неконечные точки/пиксели и Z<=0 дают NaN. Входы (N,3),(N,2), иначе ValueError. Совпадение репроекции проверяет геометрию вычислений, но не доказывает, что подобраны верные физические соответствия. EN: Return Euclidean left-camera reprojection errors, handling invalid data and nonpositive depth with NaN. Validate shapes. Explain why zero reprojection error alone does not prove correct physical correspondences.

In [ ]:
def reprojection_errors(points,pixels,calibration):  # RU: измеряем ошибку обратной проекции; EN: measure reprojection error.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте функцию; EN: implement the function.

In [ ]:
score=0;checks=[]  # RU: начинаем независимое начисление баллов; EN: initialize independent scoring.
try:  # RU: проверяем задание 1 независимо; EN: test task 1 independently.
    c=dict(fx=calib['f'],fy=calib['f'],cx=calib['cx'],cy=calib['cy'],doffs=calib['doffs'],baseline_m=calib['baseline_m'])  # RU: берём реальную калибровку; EN: use real calibration.
    old=dict(c);scaled=scale_calibration(c,.5,.75)  # RU: задаём явное изменение координат; EN: specify a coordinate change.
    assert c==old and scaled['baseline_m']==c['baseline_m']  # RU: сохраняем исходник и метры; EN: preserve input and metre units.
    assert np.allclose([scaled['fx'],scaled['cx'],scaled['doffs']],[c['fx']*.5,c['cx']*.5,c['doffs']*.5])  # RU: проверяем горизонтальный масштаб; EN: verify horizontal scaling.
    assert np.allclose([scaled['fy'],scaled['cy']],[c['fy']*.75,c['cy']*.75])  # RU: проверяем вертикальный масштаб; EN: verify vertical scaling.
    score+=25;checks.append({'task':1,'passed':True})  # RU: начисляем баллы за верный блок; EN: award points for the correct block.
except Exception as exc:  # RU: сохраняем остальные баллы при ошибке; EN: preserve other points on failure.
    checks.append({'task':1,'passed':False,'error':str(exc)})  # RU: записываем причину ошибки; EN: record the failure reason.
try:  # RU: проверяем задание 2 независимо; EN: test task 2 independently.
    d=raw.astype(np.float32)/16;report=bad_pixel_report(d,gt,2)  # RU: оцениваем реальные измерения; EN: evaluate real measurements.
    mask=np.isfinite(gt)&(gt>0);ok=mask&(d>0)&np.isfinite(d)  # RU: независимо задаём область оценки; EN: independently define evaluation regions.
    assert report['gt_count']==int(mask.sum()) and report['valid_count']==int(ok.sum())  # RU: сверяем покрытие; EN: verify coverage.
    assert report['bad_count']==int((mask&(~ok|(np.abs(d-gt)>2))).sum())  # RU: проверяем учёт пропусков; EN: verify missing-value penalties.
    perfect=bad_pixel_report(gt,gt,0);assert perfect['bad_count']==0 and perfect['bad_fraction']==0  # RU: идеальная реальная карта не содержит ошибок; EN: a perfect real map has no errors.
    score+=25;checks.append({'task':2,'passed':True})  # RU: начисляем баллы за верный блок; EN: award points for the correct block.
except Exception as exc:  # RU: сохраняем остальные баллы при ошибке; EN: preserve other points on failure.
    checks.append({'task':2,'passed':False,'error':str(exc)})  # RU: записываем причину ошибки; EN: record the failure reason.
try:  # RU: проверяем задание 3 независимо; EN: test task 3 independently.
    v,u=np.nonzero(np.isfinite(gt)&(gt>0));v=v[::997];u=u[::997]  # RU: выбираем реальные GT-соответствия; EN: sample real GT correspondences.
    left_uv=np.column_stack((u,v));right_uv=np.column_stack((u-gt[v,u],v))  # RU: строим правые координаты по эталонной disparity; EN: derive right coordinates from reference disparity.
    c=dict(fx=calib['f'],fy=calib['f'],cx=calib['cx'],cy=calib['cy'],doffs=calib['doffs'],baseline_m=calib['baseline_m'])  # RU: используем реальные параметры; EN: use real parameters.
    points=triangulate_rectified(left_uv,right_uv,c)  # RU: вычисляем точки; EN: compute points.
    assert np.allclose(points[:,2],c['fx']*c['baseline_m']/(gt[v,u]+c['doffs']))  # RU: проверяем масштаб глубины; EN: verify depth scale.
    assert np.allclose(points[:,0]/points[:,2]*c['fx']+c['cx'],u) and np.allclose(points[:,1]/points[:,2]*c['fy']+c['cy'],v)  # RU: проверяем проекцию в левую камеру; EN: verify left-camera projection.
    score+=25;checks.append({'task':3,'passed':True})  # RU: начисляем баллы за верный блок; EN: award points for the correct block.
except Exception as exc:  # RU: сохраняем остальные баллы при ошибке; EN: preserve other points on failure.
    checks.append({'task':3,'passed':False,'error':str(exc)})  # RU: записываем причину ошибки; EN: record the failure reason.
try:  # RU: проверяем задание 4 независимо; EN: test task 4 independently.
    v,u=np.nonzero(np.isfinite(gt)&(gt>0));v=v[::997];u=u[::997]  # RU: выбираем реальные пиксели с GT; EN: sample real pixels with GT.
    z=calib['f']*calib['baseline_m']/(gt[v,u]+calib['doffs'])  # RU: независимо вычисляем контрольную глубину; EN: independently compute reference depth.
    points=np.column_stack(((u-calib['cx'])*z/calib['f'],(v-calib['cy'])*z/calib['f'],z));pixels=np.column_stack((u,v))  # RU: строим контрольные лучи; EN: construct reference rays.
    c=dict(fx=calib['f'],fy=calib['f'],cx=calib['cx'],cy=calib['cy'])  # RU: выбираем параметры левой камеры; EN: select left-camera intrinsics.
    assert np.nanmax(reprojection_errors(points,pixels,c))<1e-3  # RU: проверяем обратимость на реальных данных; EN: verify invertibility on real data.
    points[0,2]=0;assert np.isnan(reprojection_errors(points,pixels,c)[0])  # RU: запрещаем деление на нулевую глубину; EN: reject zero-depth division.
    score+=25;checks.append({'task':4,'passed':True})  # RU: начисляем баллы за верный блок; EN: award points for the correct block.
except Exception as exc:  # RU: сохраняем остальные баллы при ошибке; EN: preserve other points on failure.
    checks.append({'task':4,'passed':False,'error':str(exc)})  # RU: записываем причину ошибки; EN: record the failure reason.
print(checks)  # RU: показываем результат каждого блока; EN: show each block result.
print(json.dumps({'name':(student_first+' '+student_last).strip(),'group':student_group,'assignment':assignment_id,'score':score,'max_score':100},ensure_ascii=False))  # RU: печатаем финальный JSON для проверяющего; EN: print final JSON for the checker.